## Environment Setup & Imports

If you are running this notebook on **Databricks**, uncomment and run the following line in the cell below to install the local `landseg` package in your cluster's Python environment:
```python
# %pip install -e ..
```

In [ ]:
# Databricks Setup

import os
import sys
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
import landseg.adapters.api as api

# Check if running in a Databricks environment
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Define the experiment root path where artifacts and results will be stored.
# Relative paths are standard for local repositories. On Databricks, consider utilizing
# absolute paths on a Unity Catalog Volume or DBFS path (e.g. '/Volumes/my_catalog/my_schema/my_volume/experiment/').
EXP_ROOT = '../experiment/'

if IS_DATABRICKS:
    # EXP_ROOT = '/Volumes/my_catalog/my_schema/my_volume/experiment/'
    print("Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.")

## Configure Data Harmonization (ETL) and Run

In [ ]:
# NOTE
# This is a demo script to show how to configure the data harmonization pipeline
# The purpose is to reproject and stack raw rasters into canonical Virtual Rasters (.vrt)
# The output artifacts are saved under artifacts/harmonized/run_XXXX/
# Typically this is run once when raw rasters arrive or when adjusting spatial canvas properties
# Adjust the parameters and file paths as needed for your specific use case
# Relative file paths are used here for demonstration (data shipped with the codebase)
# Use absolute file paths to ensure consistency across different environments

# init configurator
configurator = api.DataHarmonizationConfigurator(EXP_ROOT, dataset_name='sample_data')

# set target spatial canvas (CRS, resolution, and extent reference)
configurator.set_canvas(
    # EPSG code for target spatial projection
    target_crs='EPSG:3161',
    # target spatial pixel size resolution (metres)
    target_resolution=20.0,
    # path to spatial extent reference raster
    reference_raster=os.path.join(EXP_ROOT, 'input/extent_reference/sample_extent.tif')
)

# set dataset manifest containing metadata for all input rasters
configurator.set_dataset_manifest(
    dataset_manifest=os.path.join(EXP_ROOT, 'input/raw_data/manifest.json'),
    dataset_name='sample_data'
)

# set resampling algorithms for continuous and categorical rasters
configurator.set_resampling(
    continuous='bilinear',
    categorical='nearest'
)

# run data-harmonize pipeline
report = api.run(configurator.running_root_config)
print('ETL Report Summary:', report)

## Configure Data Ingestion and Run

In [ ]:
# NOTE
# This is a demo script to show how to configure the data ingestion pipeline
# The purpose is to process harmonized rasters into stable artifacts (.npz files) mapped to the grid
# The data artifacts are cataloged for access during data preparation; training will not read them
# Typically this is run once unless input data is changed or you want to adjust the settings
# Adjust the parameters and file paths as needed for your specific use case

# init configurator
configurator = api.DataIngestionConfigurator(EXP_ROOT, dataset_name='sample_data')

# set whether to force rebuild ingestion data artifacts (default: False)
configurator.set_rebuild(False)

# set targeted harmonization run (None for latest run, or specify run index/folder, e.g., 1 or 'run_0001')
configurator.set_harmonization_run(None)

# set grid parameters for tiling the input data
configurator.set_grid(
    # EPSG code for the grid CRS (must match harmonized rasters)
    crs='EPSG:3161',
    # size of the tiles to be generated in pixels - square tiles
    tile_size=256,
    # used for data augmentation and to avoid edge effects during training
    tile_overlap=128
)

# run data ingestion
api.run(configurator.running_root_config)

## Configure Data Preparation and Run

In [ ]:
# NOTE
# Data preparation is the intermediate step between data ingestion and model training
# Main processes include:
# 1. partitioning the model development data into training and validation sets
# 2. performing data augmentation if configured
# 3. normalize data and generate training-scoped artifacts (.npz files) and data schema artifacts
# Once this step is run, the training pipeline can be run directly from the artifacts
# Re-run if you want to adjust how the training view the data at runtime

# init configurator
configurator = api.DataPreparationConfigurator(EXP_ROOT)

# set whether to force rebuild preparation data artifacts (default: False)
configurator.set_rebuild(False)

# set partitioning strategy for model development data
# partitioning will be done on the tiles from non-overlapping grid blocks
configurator.set_partition(
    # percentage of dev tiles to be used for validation, rest will be used for training
    validation_blocks_ratio=0.15,
    # percentage of dev tiles to be held out for testing
    # if test holdout data is provided, this parameter will be ignored and the holdout data will be used for testing
    test_holdout_blocks_ratio=0.1
)

# set data augmentation and sampling strategy for oversampling under-represented classes in the training data
# augemntation will source from overlapping tiles and will be skipped if tile_overlap is set to 0 in the grid configuration
# set target_head to None or leave reward_classes dictionary empty for no oversampling
configurator.set_oversampling(
    target_head=None,
    # class ID: score value for oversampling (higher value means more oversampling)
    # make sure the class ID matches the label values in the training data
    reward_classes={}
)

# run data preparation
api.run(configurator.running_root_config)